# PG-DBN: Physics-Guided Dual-Branch Network for Monotonic Turbofan RUL Prediction
**Kaggle-ready, single-notebook version** of the original two Colab notebooks
(`01_Data_Pipeline_and_Baselines` + `02_Proposed_PGDBN_Architecture`), revised for the MPCON submission.

## How to run on Kaggle
1. **Add data:** right panel → *Add Input* → search **`nasa-cmaps`** (by *behrad3d*) → Add.
   Any dataset that contains `train_FD001.txt … RUL_FD004.txt` works; the notebook finds the files anywhere under `/kaggle/input`.
2. **Accelerator:** *Settings → Accelerator → GPU T4 x2* (avoid P100: recent PyTorch builds no longer support it).
3. **Run:** *Save Version → Save & Run All (Commit)*. It runs in the background (about 1–2 h on a T4) and keeps every output.
4. **Download:** when finished, open the version → *Output* → download **`pgdbn_outputs.zip`** (all tables, CSVs, figures, predictions).

Set `QUICK = True` in the next cell for a ~5-minute smoke test (FD001, one seed, few epochs).

## What changed from the original notebooks
| # | Change | Why |
|---|---|---|
| 1 | Merged into one notebook; Kaggle paths; no Google Drive | Kaggle has no Drive; one session avoids a cache handoff |
| 2 | **Same units everywhere (MVR/RVR in %)**, full precision | Notebook 1 stored fractions and the table rounded them before ×100, which is why baseline RVR showed values like 43.0 ± 0.0 |
| 3 | **Every per-engine prediction saved** (last window + full trajectory, every model and seed) | Needed for the statistical tests; also makes every number re-checkable |
| 4 | **PG-DBN trajectory RMSE before and after COPC**, and COPC applied post hoc to **every baseline** (new table) | The paper's ratchet-effect argument was missing PG-DBN's own number |
| 5 | **Deployed last-window RMSE** (after COPC) reported next to the raw-network RMSE | The deployed value at the last cycle is the running minimum, not the raw output |
| 6 | **Statistical tests**: Welch t-test over seeds, paired Wilcoxon over engines with Holm correction, bootstrap 95% CIs, Friedman + Nemenyi ranks | The accuracy claims need significance support |
| 7 | **Ablation over 3 seeds and several subsets** (rows 4 and 5 now come from the *same* trained model) | The original ablation was one seed on FD001 only |
| 8 | **Parameter count, size and CPU latency** measured in-notebook | Table IV had no code behind it |

The model, preprocessing, baselines, hyper-parameters and loss are **unchanged**.

In [ ]:
# Kaggle images ship xgboost / lightgbm; install only if missing.
import importlib, subprocess, sys
for pkg in ['xgboost', 'lightgbm']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

In [ ]:
import os, glob, zipfile, json, urllib.request, random, time, warnings, math, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.cluster import KMeans
from scipy import stats
warnings.filterwarnings('ignore')

# ============================ RUN CONFIGURATION ============================
QUICK = False                      # True -> ~5 min smoke test (FD001, 1 seed, few epochs)

DATASETS = ['FD001'] if QUICK else ['FD001', 'FD002', 'FD003', 'FD004']
SEEDS    = [0] if QUICK else [0, 1, 2]
EPOCHS_BASE = 10 if QUICK else 80  # baselines (as in the paper)
EPOCHS_PG   = 12 if QUICK else 90  # PG-DBN (as in the paper)
LAMBDA_MONO = 0.5

# Ablation: the paper's single-seed FD001 ablation, extended to 3 seeds and the
# multi-condition subsets. Remove subsets here to save time.
ABLATION_SUBSETS = ['FD001'] if QUICK else ['FD001', 'FD002', 'FD004']
ABLATION_SEEDS   = [0] if QUICK else [0, 1, 2]

N_BOOT = 2000                      # bootstrap resamples for confidence intervals
# Internal test switch (tiny synthetic run); leave as is.
if os.environ.get('PGDBN_SMOKE') == '1':
    DATASETS = ['FD001', 'FD002', 'FD003', 'FD004']; SEEDS = [0, 1]
    EPOCHS_BASE = EPOCHS_PG = 2; ABLATION_SUBSETS = ['FD001']; ABLATION_SEEDS = [0, 1]; N_BOOT = 200
# ===========================================================================

CFG = dict(window=30, rul_cap=125, val_frac=0.2, split_seed=42)
CAP, WINDOW = CFG['rul_cap'], CFG['window']
N_COND = {'FD001': 1, 'FD002': 6, 'FD003': 1, 'FD004': 6}
SENSORS = ['s2','s3','s4','s7','s8','s9','s11','s12','s13','s14','s15','s17','s20','s21']
SETTINGS = ['set1', 'set2', 'set3']
COLS = ['unit', 'cycle'] + SETTINGS + [f's{i}' for i in range(1, 22)]

ON_KAGGLE = os.path.exists('/kaggle/working')
ROOT = '/kaggle/working/pgdbn_outputs' if ON_KAGGLE else './pgdbn_outputs'
RES, FIG, TAB, PRED = [f'{ROOT}/{d}' for d in ['results', 'figures', 'tables', 'predictions']]
for d in [RES, FIG, TAB, PRED]:
    os.makedirs(d, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 10, 'font.family': 'serif',
                     'axes.labelsize': 11, 'axes.titlesize': 11, 'legend.fontsize': 9})
print('Device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU', '| outputs ->', ROOT)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

T_START = time.time()

## 1. Locate C-MAPSS
Searches, in order: `$PGDBN_DATA_DIR`, everything under `/kaggle/input`, and `./data`.
If nothing is found and the session has internet, it downloads from the NASA PCoE mirror.

In [ ]:
SEARCH_ROOTS = [os.environ.get('PGDBN_DATA_DIR', ''), '/kaggle/input', './data']

def locate_data_dir():
    for root in [r for r in SEARCH_ROOTS if r and os.path.exists(r)]:
        hits = glob.glob(f'{root}/**/train_FD001.txt', recursive=True)
        if hits:
            return os.path.dirname(hits[0])
    return None

DATA_DIR = locate_data_dir()
if DATA_DIR is None:
    os.makedirs('./data', exist_ok=True)
    url = 'https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip'
    try:
        print('Not found locally, downloading', url)
        urllib.request.urlretrieve(url, './data/cmapss.zip')
        for _ in range(3):  # archive contains a nested zip
            for z in glob.glob('./data/**/*.zip', recursive=True):
                with zipfile.ZipFile(z) as f: f.extractall(os.path.splitext(z)[0])
        DATA_DIR = locate_data_dir()
    except Exception as e:
        print('Download failed:', e)
assert DATA_DIR, ('C-MAPSS not found. On Kaggle: Add Input -> "nasa-cmaps" (behrad3d). '
                  'Elsewhere: put train_FD00x.txt / test_FD00x.txt / RUL_FD00x.txt under ./data')
print('Data directory:', DATA_DIR)

def read_split(fd):
    rd = lambda n: pd.read_csv(f'{DATA_DIR}/{n}', sep=r'\s+', header=None, names=COLS)
    tr, te = rd(f'train_{fd}.txt'), rd(f'test_{fd}.txt')
    rul_end = pd.read_csv(f'{DATA_DIR}/RUL_{fd}.txt', sep=r'\s+', header=None)[0].values
    return tr, te, rul_end

rows = []
for fd in DATASETS:
    tr, te, r = read_split(fd)
    L = tr.groupby('unit').cycle.max()
    rows.append(dict(subset=fd, train_engines=tr.unit.nunique(), test_engines=te.unit.nunique(),
                     train_rows=len(tr), min_life=L.min(), max_life=L.max(), conditions=N_COND[fd]))
pd.DataFrame(rows)

## 2. Exploratory plots (paper Fig. 1 and Fig. 2)

In [ ]:
if 'FD001' in DATASETS:
    tr, _, _ = read_split('FD001')
    fig, axes = plt.subplots(2, 4, figsize=(12, 5)); axes = axes.ravel()
    for ax, s in zip(axes, ['s2', 's3', 's4', 's7', 's11', 's12', 's15', 's21']):
        for u in [1, 2, 3, 4, 5]:
            g = tr[tr.unit == u]
            ax.plot(g.cycle.max() - g.cycle, g[s], lw=0.8)
        ax.invert_xaxis(); ax.set_title(s); ax.set_xlabel('cycles to failure')
    plt.tight_layout(); plt.savefig(f'{FIG}/fig1_sensor_trends_FD001.pdf'); plt.savefig(f'{FIG}/fig1_sensor_trends_FD001.png'); plt.show()

fig, ax = plt.subplots(figsize=(5, 3))
t = np.arange(0, 250)
ax.plot(t, 250 - t, '--', label='Linear RUL'); ax.plot(t, np.minimum(250 - t, CAP), label=f'Piecewise RUL (cap {CAP})')
ax.set_xlabel('cycle'); ax.set_ylabel('RUL'); ax.legend(); plt.tight_layout()
plt.savefig(f'{FIG}/fig2_piecewise_rul.pdf'); plt.savefig(f'{FIG}/fig2_piecewise_rul.png'); plt.show()

## 3. Preprocessing (unchanged)
14 sensors · K-Means ($k=6$) regimes on the operating settings for FD002/FD004 · per-regime z-score ·
**K-Means and z-score statistics fitted on training engines only** · piecewise-linear RUL capped at 125 · window $W=30$.

In [ ]:
def add_rul(df, cap, rul_end=None):
    mx = df.groupby('unit')['cycle'].transform('max')
    raw = mx - df['cycle']
    if rul_end is not None:
        raw = raw + df['unit'].map(dict(zip(range(1, len(rul_end) + 1), rul_end)))
    df['RUL_raw'] = raw.astype(float); df['RUL'] = np.minimum(raw, cap).astype(float)
    return df

def normalize(fit_df, dfs, n_cond):
    if n_cond > 1:
        km = KMeans(n_cond, n_init=10, random_state=0).fit(fit_df[SETTINGS].values)
        for d in [fit_df] + dfs: d['cond'] = km.predict(d[SETTINGS].values)
    else:
        for d in [fit_df] + dfs: d['cond'] = 0
    mu = fit_df.groupby('cond')[SENSORS].mean(); sd = fit_df.groupby('cond')[SENSORS].std() + 1e-8
    out = []
    for d in [fit_df] + dfs:
        d = d.copy()
        for c in mu.index:
            m = d.cond == c
            d.loc[m, SENSORS] = (d.loc[m, SENSORS] - mu.loc[c]) / sd.loc[c]
        out.append(d)
    return out

def make_windows(df, W, last_only=False):
    X, y, yraw, U, C = [], [], [], [], []
    for unit, g in df.groupby('unit'):
        a = g[SENSORS].values.astype(np.float32); r = g['RUL'].values; rr = g['RUL_raw'].values; cy = g['cycle'].values
        if len(a) < W:
            p = W - len(a)
            a = np.vstack([np.repeat(a[:1], p, 0), a])
            r, rr, cy = [np.concatenate([np.full(p, v[0]), v]) for v in (r, rr, cy)]
        ends = [len(a)] if last_only else range(W, len(a) + 1)
        for e in ends:
            X.append(a[e - W:e]); y.append(r[e - 1]); yraw.append(rr[e - 1]); U.append(unit); C.append(cy[e - 1])
    return (np.stack(X), np.array(y, np.float32), np.array(yraw, np.float32), np.array(U), np.array(C))

def preprocess(fd):
    tr, te, rul_end = read_split(fd)
    tr = add_rul(tr, CAP); te = add_rul(te, CAP, rul_end)
    units = tr.unit.unique(); rng = np.random.RandomState(CFG['split_seed'])
    va_units = rng.choice(units, int(len(units) * CFG['val_frac']), replace=False)
    trn, val = tr[~tr.unit.isin(va_units)].copy(), tr[tr.unit.isin(va_units)].copy()
    trn, val, te = normalize(trn, [val, te], N_COND[fd])
    out = {}
    for name, d, last in [('tr', trn, False), ('va', val, False), ('te', te, True), ('tf', te, False)]:
        X, y, yraw, U, C = make_windows(d, WINDOW, last)
        out.update({f'X{name}': X, f'y{name}': y, f'yraw{name}': yraw, f'u{name}': U, f'c{name}': C})
    print(f"{fd}: train {out['Xtr'].shape}, val {out['Xva'].shape}, test-last {out['Xte'].shape}, test-full {out['Xtf'].shape}")
    return out

DATA = {fd: preprocess(fd) for fd in DATASETS}

# Sanity check: the last trajectory window of each test engine is the last-window input.
for fd, D in DATA.items():
    last_idx = np.array([np.where(D['utf'] == u)[0][np.argmax(D['ctf'][D['utf'] == u])] for u in D['ute']])
    assert np.allclose(D['Xtf'][last_idx], D['Xte']) and np.allclose(D['ytf'][last_idx], D['yte'])
    D['last_idx'] = last_idx
print('Last-window / trajectory alignment verified.')

## 4. Metrics and COPC
* **RMSE** and **NASA score** on the last test window (as in the literature). Reported twice:
  *raw network output* (paper Table I convention) and *deployed* (after COPC, i.e. the value actually reported at the last cycle).
* Along full test trajectories: **trajectory RMSE**, **MVR** (% of steps rising > 0.5 cycles), **RVR** (% of steps falling > 1.5 cycles),
  **roughness** (mean |second difference| per engine, averaged over engines). **All rates are in percent.**
* Each trajectory metric is computed on the **raw** output and on the **COPC** output, for every model.

In [ ]:
def rmse(p, t): return float(np.sqrt(np.mean((p - t) ** 2)))
def nasa_score(p, t):
    d = p - t
    return float(np.sum(np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0)))

def copc(pred, U, C):
    # Causal Operational Physics Constraint: per-engine running minimum in cycle order (zero look-ahead).
    out = np.empty_like(pred)
    for u in np.unique(U):
        idx = np.where(U == u)[0]; idx = idx[np.argsort(C[idx])]
        out[idx] = np.minimum.accumulate(pred[idx])
    return out

def consistency(pred, U, C, tol=0.5):
    mv = rv = n = 0; rough = []
    for u in np.unique(U):
        m = U == u; o = np.argsort(C[m]); p = pred[m][o]; c = C[m][o]
        d = np.diff(p)[np.diff(c) == 1]
        mv += np.sum(d > tol); rv += np.sum(d < -1.0 - tol); n += len(d)
        if len(p) > 2: rough.append(np.mean(np.abs(np.diff(p, 2))))
    return dict(MVR=100.0 * mv / max(n, 1), RVR=100.0 * rv / max(n, 1), Rough=float(np.mean(rough)) if rough else 0.0)

def per_engine_traj_rmse(pred, D):
    return np.array([rmse(pred[D['utf'] == u], D['ytf'][D['utf'] == u]) for u in D['ute']])

def evaluate(pred_last, pred_traj_raw, D):
    # pred_last: raw last-window predictions; pred_traj_raw: raw predictions on every test window.
    pc = copc(pred_traj_raw, D['utf'], D['ctf'])
    dep_last = pc[D['last_idx']]
    raw_c, cop_c = consistency(pred_traj_raw, D['utf'], D['ctf']), consistency(pc, D['utf'], D['ctf'])
    return dict(
        RMSE=rmse(pred_last, D['yte']), Score=nasa_score(pred_last, D['yte']),
        RMSE_dep=rmse(dep_last, D['yte']), Score_dep=nasa_score(dep_last, D['yte']),
        TrajRMSE_raw=rmse(pred_traj_raw, D['ytf']), TrajRMSE_copc=rmse(pc, D['ytf']),
        MVR_raw=raw_c['MVR'], RVR_raw=raw_c['RVR'], Rough_raw=raw_c['Rough'],
        MVR_copc=cop_c['MVR'], RVR_copc=cop_c['RVR'], Rough_copc=cop_c['Rough'])

## 5. Baseline architectures (unchanged)

In [ ]:
class MLP(nn.Module):
    def __init__(self, W, F, drop=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(W * F, 256), nn.ReLU(), nn.Dropout(drop),
                                 nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

class CNN1D(nn.Module):
    def __init__(self, W, F, drop=0.2):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(F, 32, 5, padding=2), nn.ReLU(), nn.Conv1d(32, 64, 5, padding=2), nn.ReLU(),
                                  nn.Conv1d(64, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool1d(1))
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(drop), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x): return self.head(self.conv(x.transpose(1, 2))).squeeze(-1)

class Chomp1d(nn.Module):
    def __init__(self, n): super().__init__(); self.n = n
    def forward(self, x): return x[:, :, :-self.n].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, in_c, out_c, k, dilation, padding, drop=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.Conv1d(in_c, out_c, k, padding=padding, dilation=dilation), Chomp1d(padding), nn.ReLU(), nn.Dropout(drop),
                                 nn.Conv1d(out_c, out_c, k, padding=padding, dilation=dilation), Chomp1d(padding), nn.ReLU(), nn.Dropout(drop))
        self.downsample = nn.Conv1d(in_c, out_c, 1) if in_c != out_c else None
        self.relu = nn.ReLU()
    def forward(self, x):
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(self.net(x) + res)

class TCN(nn.Module):
    def __init__(self, W, F, num_channels=(32, 64, 64), k=3, drop=0.2):
        super().__init__()
        layers = [TemporalBlock(F if i == 0 else num_channels[i - 1], c, k, 2 ** i, (k - 1) * 2 ** i, drop)
                  for i, c in enumerate(num_channels)]
        self.network = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Linear(num_channels[-1], 32), nn.ReLU(), nn.Dropout(drop), nn.Linear(32, 1))
    def forward(self, x): return self.head(self.network(x.transpose(1, 2))[:, :, -1]).squeeze(-1)

class LSTMReg(nn.Module):
    def __init__(self, W, F, h=64, layers=2, drop=0.2):
        super().__init__()
        self.lstm = nn.LSTM(F, h, layers, batch_first=True, dropout=drop)
        self.head = nn.Sequential(nn.Linear(h, 32), nn.ReLU(), nn.Dropout(drop), nn.Linear(32, 1))
    def forward(self, x):
        o, _ = self.lstm(x); return self.head(o[:, -1]).squeeze(-1)

class BiLSTMAttention(nn.Module):
    def __init__(self, W, F, h=32, layers=2, drop=0.2):
        super().__init__()
        self.lstm = nn.LSTM(F, h, layers, batch_first=True, dropout=drop, bidirectional=True)
        self.attention = nn.Sequential(nn.Linear(h * 2, 32), nn.Tanh(), nn.Linear(32, 1))
        self.head = nn.Sequential(nn.Linear(h * 2, 32), nn.ReLU(), nn.Dropout(drop), nn.Linear(32, 1))
    def forward(self, x):
        o, _ = self.lstm(x)
        alpha = torch.softmax(self.attention(o).squeeze(-1), dim=-1).unsqueeze(-1)
        return self.head(torch.sum(o * alpha, dim=1)).squeeze(-1)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model); pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class TemporalTransformer(nn.Module):
    def __init__(self, W, F, d_model=64, nhead=4, num_layers=2, drop=0.2):
        super().__init__()
        self.proj = nn.Linear(F, d_model); self.pos_enc = PositionalEncoding(d_model, max_len=W)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=128, dropout=drop,
                                           activation='relu', batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Dropout(drop), nn.Linear(32, 1))
    def forward(self, x): return self.head(self.transformer(self.pos_enc(self.proj(x))).mean(dim=1)).squeeze(-1)

@torch.no_grad()
def predict(model, X, bs=2048):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        out.append(model(torch.as_tensor(X[i:i + bs], device=DEVICE)).cpu().numpy())
    return np.clip(np.concatenate(out) * CAP, 0, CAP)

def train_nn(model, D, epochs=EPOCHS_BASE, bs=256, lr=1e-3, patience=12):
    model.to(DEVICE)
    Xtr = torch.as_tensor(D['Xtr'], device=DEVICE); ytr = torch.as_tensor(D['ytr'] / CAP, device=DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=5)
    best, best_state, wait, n_ep = 1e9, None, 0, 0
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr), device=DEVICE); n_ep += 1
        for i in range(0, len(perm), bs):
            idx = perm[i:i + bs]
            loss = nn.functional.mse_loss(model(Xtr[idx]), ytr[idx])
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        v = rmse(predict(model, D['Xva']), D['yva']); sched.step(v)
        if v < best: best, wait, best_state = v, 0, {k: x.detach().clone() for k, x in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state)
    return model, n_ep

In [ ]:
import xgboost as xgb
import lightgbm as lgb

def window_features(X):
    t = np.arange(X.shape[1]); t = (t - t.mean()) / ((t - t.mean()) ** 2).sum()
    slope = np.einsum('t,ntf->nf', t, X)
    return np.concatenate([X[:, -1], X.mean(1), X.std(1), slope], 1)

def run_xgb(D, seed):
    m = xgb.XGBRegressor(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                         random_state=seed, tree_method='hist', early_stopping_rounds=50,
                         device='cuda' if DEVICE == 'cuda' else 'cpu')
    m.fit(window_features(D['Xtr']), D['ytr'], eval_set=[(window_features(D['Xva']), D['yva'])], verbose=False)
    f = lambda X: np.clip(m.predict(window_features(X)), 0, CAP)
    return f(D['Xte']), f(D['Xtf'])

def run_lgb(D, seed):
    m = lgb.LGBMRegressor(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8, subsample_freq=1,
                          colsample_bytree=0.8, random_state=seed, verbosity=-1, n_jobs=-1)
    m.fit(window_features(D['Xtr']), D['ytr'], eval_set=[(window_features(D['Xva']), D['yva'])],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    f = lambda X: np.clip(m.predict(window_features(X)), 0, CAP)
    return f(D['Xte']), f(D['Xtf'])

## 6. Train the 8 baselines (subsets × seeds)

In [ ]:
NN_MODELS = {'MLP': MLP, '1D-CNN': CNN1D, 'TCN': TCN, 'LSTM': LSTMReg,
             'BiLSTM-Attn': BiLSTMAttention, 'Transformer': TemporalTransformer}
BASELINES = list(NN_MODELS) + ['XGBoost', 'LightGBM']
PROPOSED = 'PG-DBN'
MODEL_ORDER = BASELINES + [PROPOSED]

results = []          # one row per (subset, model, seed)
PREDS = {}            # PREDS[(fd, model, seed)] = dict(last=..., traj=...)  (raw outputs)

def record(fd, name, seed, pl, pf, n_ep, t0):
    PREDS[(fd, name, seed)] = dict(last=pl.astype(np.float32), traj=pf.astype(np.float32))
    r = dict(dataset=fd, model=name, seed=seed, epochs=n_ep, time_s=time.time() - t0, **evaluate(pl, pf, DATA[fd]))
    results.append(r)
    print(f"[{fd}] {name:<12} seed {seed} | RMSE {r['RMSE']:6.2f} | Score {r['Score']:8.1f} | "
          f"MVR {r['MVR_raw']:5.1f}% | TrajRMSE raw {r['TrajRMSE_raw']:6.2f} -> COPC {r['TrajRMSE_copc']:6.2f} | {r['time_s']:.0f}s")

for fd in DATASETS:
    D = DATA[fd]; W, F = D['Xtr'].shape[1:]
    for seed in SEEDS:
        for name, cls in NN_MODELS.items():
            set_seed(seed); t0 = time.time()
            model, n_ep = train_nn(cls(W, F), D)
            record(fd, name, seed, predict(model, D['Xte']), predict(model, D['Xtf']), n_ep, t0)
        set_seed(seed); t0 = time.time(); pl, pf = run_xgb(D, seed); record(fd, 'XGBoost', seed, pl, pf, 0, t0)
        set_seed(seed); t0 = time.time(); pl, pf = run_lgb(D, seed); record(fd, 'LightGBM', seed, pl, pf, 0, t0)

pd.DataFrame(results).to_csv(f'{RES}/baseline_results_per_seed.csv', index=False)
print(f'Baselines done in {(time.time() - T_START) / 60:.1f} min')

## 7. Proposed PG-DBN (architecture, loss and training unchanged)
Pairs of consecutive windows from the same engine · loss $= \tfrac12[\mathrm{MSE}_t+\mathrm{MSE}_{t+1}] + \lambda\,\mathbb{E}[\max(0,\hat y_{t+1}-\hat y_t)^2]$, $\lambda=0.5$ ·
COPC applied only at inference. Raw outputs are stored; COPC is applied inside `evaluate`.

In [ ]:
class TrajectoryPairDataset(Dataset):
    def __init__(self, X, y, units, cycles):
        pairs = []
        for u in np.unique(units):
            idx = np.where(units == u)[0]; idx = idx[np.argsort(cycles[idx])]; c = cycles[idx]
            pairs += [(idx[i], idx[i + 1]) for i in range(len(idx) - 1) if c[i + 1] - c[i] == 1]
        self.pairs = pairs
        self.X = torch.as_tensor(X, dtype=torch.float32); self.y = torch.as_tensor(y / CAP, dtype=torch.float32)
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        a, b = self.pairs[i]
        return self.X[a], self.y[a], self.X[b], self.y[b]

def trend_features(x):
    mean = x.mean(dim=1)
    t = torch.linspace(-1.0, 1.0, x.shape[1], device=x.device).view(1, -1, 1)
    slope = torch.sum(t * (x - mean.unsqueeze(1)), dim=1) / (torch.sum(t ** 2) + 1e-7)
    return torch.cat([x[:, -1, :], mean, x.std(dim=1), slope], dim=1)

class PGDBN(nn.Module):
    def __init__(self, W=30, F=14, lstm_h=64, stat_h=64, drop=0.2):
        super().__init__()
        self.lstm = nn.LSTM(F, lstm_h, num_layers=2, batch_first=True, dropout=drop)
        self.trend_mlp = nn.Sequential(nn.Linear(4 * F, stat_h), nn.ReLU(), nn.Dropout(drop * 0.5),
                                       nn.Linear(stat_h, stat_h), nn.ReLU())
        self.ln = nn.LayerNorm(lstm_h + stat_h)
        self.head = nn.Sequential(nn.Linear(lstm_h + stat_h, 32), nn.ReLU(), nn.Dropout(drop * 0.5), nn.Linear(32, 1))
    def forward(self, x):
        o, _ = self.lstm(x)
        h = torch.cat([o[:, -1, :], self.trend_mlp(trend_features(x))], dim=1)
        return self.head(self.ln(h)).squeeze(-1)

class PhysicsGuidedMSELoss(nn.Module):
    def __init__(self, lambda_mono=LAMBDA_MONO):
        super().__init__(); self.lm = lambda_mono; self.mse = nn.MSELoss()
    def forward(self, p1, y1, p2, y2):
        loss = 0.5 * (self.mse(p1, y1) + self.mse(p2, y2))
        if self.lm > 0:
            loss = loss + self.lm * torch.mean(torch.relu(p2 - p1) ** 2)
        return loss

def train_pgdbn(model, D, loss_fn, epochs=EPOCHS_PG, bs=256, lr=1e-3, patience=14):
    model.to(DEVICE)
    loader = DataLoader(TrajectoryPairDataset(D['Xtr'], D['ytr'], D['utr'], D['ctr']), batch_size=bs, shuffle=True, drop_last=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5)
    best, best_state, wait, n_ep = 1e9, None, 0, 0
    for ep in range(epochs):
        model.train(); n_ep += 1
        for x1, y1, x2, y2 in loader:
            x1, y1, x2, y2 = x1.to(DEVICE), y1.to(DEVICE), x2.to(DEVICE), y2.to(DEVICE)
            loss = loss_fn(model(x1), y1, model(x2), y2)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        v = rmse(predict(model, D['Xva']), D['yva']); sched.step(v)
        if v < best: best, wait, best_state = v, 0, {k: x.detach().clone() for k, x in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state)
    return model, n_ep

In [ ]:
for fd in DATASETS:
    D = DATA[fd]; W, F = D['Xtr'].shape[1:]
    for seed in SEEDS:
        set_seed(seed); t0 = time.time()
        model, n_ep = train_pgdbn(PGDBN(W, F), D, PhysicsGuidedMSELoss(LAMBDA_MONO))
        record(fd, PROPOSED, seed, predict(model, D['Xte']), predict(model, D['Xtf']), n_ep, t0)
        if seed == 0:
            torch.save(model.state_dict(), f'{RES}/pgdbn_{fd}_seed0.pt')

RESULTS = pd.DataFrame(results)
RESULTS.to_csv(f'{RES}/all_results_per_seed.csv', index=False)

# Save every prediction so every number in the paper can be re-checked without retraining.
np.savez_compressed(f'{PRED}/predictions.npz',
                    **{f'{fd}|{m}|{s}|{k}': v for (fd, m, s), d in PREDS.items() for k, v in d.items()})
for fd, D in DATA.items():
    np.savez_compressed(f'{PRED}/ground_truth_{fd}.npz', y_last=D['yte'], units_last=D['ute'],
                        y_traj=D['ytf'], units_traj=D['utf'], cycles_traj=D['ctf'])
print(f'PG-DBN done. Elapsed {(time.time() - T_START) / 60:.1f} min')

## 8. Result tables
All values are **mean ± sample std (ddof = 1) over seeds**. Tables are written to `tables/` as CSV and LaTeX.

* **Table I (paper convention):** RMSE / NASA score of the raw network on the last window; trajectory metrics on the *raw* output for baselines and on the *COPC* output for PG-DBN.
* **Table I-b (new, fair comparison):** COPC applied to *every* model. Shows the cost of the running minimum (the ratchet effect) for each model, including PG-DBN.

In [ ]:
def ms(series, dec):
    m, s = series.mean(), (series.std(ddof=1) if len(series) > 1 else 0.0)
    return f'{m:.{dec}f} ± {s:.{dec}f}'

def summarize(df, spec):
    rows = []
    for fd in DATASETS:
        for m in MODEL_ORDER:
            g = df[(df.dataset == fd) & (df.model == m)]
            if len(g) == 0: continue
            row = {'Subset': fd, 'Model': m}
            for label, (col_base, col_prop, dec) in spec.items():
                col = col_prop if m == PROPOSED else col_base
                row[label] = ms(g[col], dec)
            rows.append(row)
    return pd.DataFrame(rows)

def save_table(df, name, caption):
    df.to_csv(f'{TAB}/{name}.csv', index=False)
    with open(f'{TAB}/{name}.tex', 'w') as f:
        f.write(df.to_latex(index=False, caption=caption, label=f'tab:{name}', escape=True))
    display(df)

TABLE1 = summarize(RESULTS, {
    'RMSE': ('RMSE', 'RMSE', 2), 'NASA Score': ('Score', 'Score', 1),
    'MVR (%)': ('MVR_raw', 'MVR_copc', 2), 'RVR (%)': ('RVR_raw', 'RVR_copc', 2),
    'Roughness': ('Rough_raw', 'Rough_copc', 2)})
save_table(TABLE1, 'table1_benchmark', 'Benchmark of nine models on C-MAPSS (mean ± std over seeds). '
           'Baseline trajectory metrics on raw outputs; PG-DBN on the COPC output.')

TABLE1B = summarize(RESULTS, {
    'Traj. RMSE raw': ('TrajRMSE_raw', 'TrajRMSE_raw', 2), 'Traj. RMSE +COPC': ('TrajRMSE_copc', 'TrajRMSE_copc', 2),
    'Last-window RMSE +COPC': ('RMSE_dep', 'RMSE_dep', 2), 'RVR +COPC (%)': ('RVR_copc', 'RVR_copc', 2),
    'Roughness +COPC': ('Rough_copc', 'Rough_copc', 2), 'Raw MVR (%)': ('MVR_raw', 'MVR_raw', 2)})
save_table(TABLE1B, 'table1b_copc_all_models', 'Every model with COPC applied post hoc. '
           'Traj. RMSE raw -> +COPC is the cost of the running minimum (ratchet effect).')

In [ ]:
# Ratchet cost per model: increase in trajectory RMSE caused by COPC (mean over seeds).
ratchet = (RESULTS.assign(Ratchet=RESULTS.TrajRMSE_copc - RESULTS.TrajRMSE_raw)
           .groupby(['dataset', 'model'])[['TrajRMSE_raw', 'TrajRMSE_copc', 'Ratchet']].mean().round(2)
           .reset_index())
ratchet['model'] = pd.Categorical(ratchet['model'], MODEL_ORDER, ordered=True)
ratchet = ratchet.sort_values(['dataset', 'TrajRMSE_copc'])
ratchet.to_csv(f'{TAB}/ratchet_cost.csv', index=False)
print('Sorted by trajectory RMSE after COPC (lower = more useful deployed trajectory):')
display(ratchet)

## 9. Ablation (multi-seed, multi-subset)
Rows 1–4 are trained; **row 5 is row 4's model with COPC applied** (the original notebook retrained it, which is why rows 4 and 5 had identical RMSE).
Last-window RMSE / score are raw-network values; MVR and roughness are on the deployed trajectory of that row.

In [ ]:
class LSTMOnly(nn.Module):
    def __init__(self, F):
        super().__init__()
        self.lstm = nn.LSTM(F, 64, 2, batch_first=True, dropout=0.2); self.head = nn.Linear(64, 1)
    def forward(self, x):
        o, _ = self.lstm(x); return self.head(o[:, -1]).squeeze(-1)

class TrendOnly(nn.Module):
    def __init__(self, F):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(4 * F, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x): return self.mlp(trend_features(x)).squeeze(-1)

ABL_CFG = [('1. LSTM only (sequence)', 'lstm', 0.0),
           ('2. Trend branch only', 'trend', 0.0),
           ('3. Dual branch, no physics', 'dbn', 0.0),
           ('4. + Mono. loss (λ=0.5)', 'dbn', LAMBDA_MONO)]
abl_rows = []
for fd in ABLATION_SUBSETS:
    D = DATA[fd]; W, F = D['Xtr'].shape[1:]
    for seed in ABLATION_SEEDS:
        for name, kind, lm in ABL_CFG:
            set_seed(seed)
            net = {'lstm': lambda: LSTMOnly(F), 'trend': lambda: TrendOnly(F), 'dbn': lambda: PGDBN(W, F)}[kind]()
            net, _ = train_pgdbn(net, D, PhysicsGuidedMSELoss(lm))
            ev = evaluate(predict(net, D['Xte']), predict(net, D['Xtf']), D)
            abl_rows.append(dict(subset=fd, seed=seed, config=name, RMSE=ev['RMSE'], Score=ev['Score'],
                                 TrajRMSE=ev['TrajRMSE_raw'], MVR=ev['MVR_raw'], Rough=ev['Rough_raw']))
            if kind == 'dbn' and lm > 0:   # row 5: same model, COPC at inference
                abl_rows.append(dict(subset=fd, seed=seed, config='5. + COPC (PG-DBN)', RMSE=ev['RMSE'], Score=ev['Score'],
                                     TrajRMSE=ev['TrajRMSE_copc'], MVR=ev['MVR_copc'], Rough=ev['Rough_copc']))
            print(f"[{fd} seed {seed}] {name:<28} RMSE {ev['RMSE']:.2f}  MVR {ev['MVR_raw']:.2f}%")

ABL = pd.DataFrame(abl_rows); ABL.to_csv(f'{RES}/ablation_per_seed.csv', index=False)
ABL_TABLE = (ABL.groupby(['subset', 'config'], sort=False)
             .agg(**{c: (c, lambda s, c=c: ms(s, 1 if c == 'Score' else 2)) for c in ['RMSE', 'Score', 'TrajRMSE', 'MVR', 'Rough']})
             .reset_index().rename(columns={'TrajRMSE': 'Traj. RMSE', 'MVR': 'MVR (%)', 'Rough': 'Roughness'}))
save_table(ABL_TABLE, 'table2_ablation', 'Ablation (mean ± std over seeds). Row 5 is row 4 with COPC at inference.')

## 10. Statistical tests
1. **Welch t-test over seeds** (n = number of seeds) on last-window RMSE, PG-DBN vs each baseline. Low power with 3 seeds; reported for completeness.
2. **Paired Wilcoxon signed-rank test over test engines** on per-engine absolute error (each model's error averaged over seeds), PG-DBN vs each baseline,
   **Holm-corrected** across the 8 baselines within each subset. Effect size = matched-pairs rank-biserial correlation *r* (negative = PG-DBN has smaller errors).
3. **Bootstrap 95% CI** of the RMSE difference (baseline − PG-DBN; positive = PG-DBN better), resampling engines, using the seed-mean RMSE.
4. **Ratchet test:** the same Wilcoxon + bootstrap on **per-engine trajectory RMSE after COPC** — does PG-DBN keep a more accurate deployed trajectory than *baseline + COPC*?
5. **Friedman test + Nemenyi critical difference** on average RMSE ranks across subsets (Demšar, JMLR 2006).

In [ ]:
def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); adj = np.empty_like(p); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (len(p) - rank) * p[i])); adj[i] = running
    return adj

def wilcoxon_paired(a, b):
    # a = PG-DBN per-engine errors, b = baseline per-engine errors. Returns p, rank-biserial r.
    d = a - b; d = d[d != 0]
    if len(d) < 5: return np.nan, np.nan
    p = stats.wilcoxon(d, alternative='two-sided').pvalue
    rk = stats.rankdata(np.abs(d)); r = (rk[d > 0].sum() - rk[d < 0].sum()) / rk.sum()
    return float(p), float(r)

def seed_stack(fd, model, kind):
    return np.stack([PREDS[(fd, model, s)][kind] for s in SEEDS])   # (n_seeds, n)

rng_boot = np.random.default_rng(0)

def boot_ci_rmse_diff(pb, pp, y, groups=None):
    # pb, pp: (n_seeds, n) predictions. groups: engine id per element (trajectory case) or None (one row per engine).
    if groups is None:
        groups = np.arange(len(y))
    uniq = np.unique(groups); members = {g: np.where(groups == g)[0] for g in uniq}
    diffs = []
    for _ in range(N_BOOT):
        pick = rng_boot.choice(uniq, len(uniq), replace=True)
        idx = np.concatenate([members[g] for g in pick])
        rb = np.mean([rmse(p[idx], y[idx]) for p in pb]); rp = np.mean([rmse(p[idx], y[idx]) for p in pp])
        diffs.append(rb - rp)
    return np.percentile(diffs, [2.5, 97.5])

stat_rows, traj_rows = [], []
for fd in DATASETS:
    D = DATA[fd]; y = D['yte']
    prop_last = seed_stack(fd, PROPOSED, 'last')
    prop_traj_copc = np.stack([copc(p, D['utf'], D['ctf']) for p in seed_stack(fd, PROPOSED, 'traj')])
    e_prop = np.abs(prop_last - y).mean(0)
    te_prop = np.mean([per_engine_traj_rmse(p, D) for p in prop_traj_copc], axis=0)
    for b in BASELINES:
        base_last = seed_stack(fd, b, 'last')
        # 1. Welch over seeds
        r_p = RESULTS[(RESULTS.dataset == fd) & (RESULTS.model == PROPOSED)].RMSE.values
        r_b = RESULTS[(RESULTS.dataset == fd) & (RESULTS.model == b)].RMSE.values
        p_welch = stats.ttest_ind(r_p, r_b, equal_var=False).pvalue if len(SEEDS) > 1 else np.nan
        # 2. Wilcoxon over engines (last window)
        p_w, r_w = wilcoxon_paired(e_prop, np.abs(base_last - y).mean(0))
        # 3. Bootstrap CI
        lo, hi = boot_ci_rmse_diff(base_last, prop_last, y)
        stat_rows.append(dict(Subset=fd, Baseline=b, RMSE_baseline=r_b.mean(), RMSE_PGDBN=r_p.mean(),
                              Diff=r_b.mean() - r_p.mean(), CI95_low=lo, CI95_high=hi,
                              p_Welch_seeds=p_welch, p_Wilcoxon=p_w, r_rank_biserial=r_w))
        # 4. Ratchet: trajectory RMSE after COPC, PG-DBN vs baseline + COPC
        base_traj_copc = np.stack([copc(p, D['utf'], D['ctf']) for p in seed_stack(fd, b, 'traj')])
        te_base = np.mean([per_engine_traj_rmse(p, D) for p in base_traj_copc], axis=0)
        p_t, r_t = wilcoxon_paired(te_prop, te_base)
        lo_t, hi_t = boot_ci_rmse_diff(base_traj_copc, prop_traj_copc, D['ytf'], groups=D['utf'])
        traj_rows.append(dict(Subset=fd, Baseline=b,
                              TrajRMSE_COPC_baseline=np.mean([rmse(p, D['ytf']) for p in base_traj_copc]),
                              TrajRMSE_COPC_PGDBN=np.mean([rmse(p, D['ytf']) for p in prop_traj_copc]),
                              CI95_low=lo_t, CI95_high=hi_t, p_Wilcoxon=p_t, r_rank_biserial=r_t))

def finish(df):
    df['p_Holm'] = np.nan
    for fd in df.Subset.unique():
        m = (df.Subset == fd) & df.p_Wilcoxon.notna()
        df.loc[m, 'p_Holm'] = holm(df.loc[m, 'p_Wilcoxon'])
    df['Significant (Holm < 0.05)'] = np.where(df.p_Holm < 0.05, 'yes', 'no')
    return df.round(4)

STATS = finish(pd.DataFrame(stat_rows)); STATS.to_csv(f'{TAB}/stats_last_window.csv', index=False)
TRAJ_STATS = finish(pd.DataFrame(traj_rows)); TRAJ_STATS.to_csv(f'{TAB}/stats_trajectory_copc.csv', index=False)
print('Last-window RMSE: PG-DBN vs each baseline (Diff and CI > 0 means PG-DBN is better)'); display(STATS)
print('Deployed trajectory RMSE after COPC: PG-DBN vs baseline + COPC'); display(TRAJ_STATS)

In [ ]:
# 5. Friedman + Nemenyi on mean last-window RMSE (rows = subsets, columns = models)
NEMENYI_Q05 = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164}
mean_rmse = RESULTS.groupby(['dataset', 'model']).RMSE.mean().unstack()[MODEL_ORDER]
if len(DATASETS) >= 3:
    chi2, p_fr = stats.friedmanchisquare(*[mean_rmse[m].values for m in MODEL_ORDER])
    ranks = mean_rmse.rank(axis=1).mean().sort_values()
    k, N = len(MODEL_ORDER), len(mean_rmse)
    CD = NEMENYI_Q05[k] * np.sqrt(k * (k + 1) / (6 * N))
    RANKS = ranks.rename('Average rank').round(2).reset_index().rename(columns={'index': 'Model', 'model': 'Model'})
    RANKS['Worse than PG-DBN by > CD?'] = np.where(RANKS['Average rank'] - ranks[PROPOSED] > CD, 'yes', 'no')
    RANKS.to_csv(f'{TAB}/friedman_ranks.csv', index=False)
    print(f'Friedman chi2 = {chi2:.2f}, p = {p_fr:.4f} | Nemenyi CD (alpha=0.05, k={k}, N={N}) = {CD:.2f}')
    display(RANKS)
    with open(f'{TAB}/friedman.txt', 'w') as f:
        f.write(f'Friedman chi2={chi2:.3f}, p={p_fr:.5f}, k={k}, N={N}, Nemenyi CD={CD:.3f}\n{RANKS.to_string(index=False)}\n')
else:
    print('Friedman test needs at least 3 subsets; skipped in QUICK mode.')

## 11. Figures

In [ ]:
fd = DATASETS[0]; D = DATA[fd]; s0 = SEEDS[0]
units = [u for u in np.unique(D['utf']) if (D['utf'] == u).sum() > 140][:3] or list(np.unique(D['utf'])[:3])
fig, axes = plt.subplots(1, len(units), figsize=(12, 3.5), sharey=True); axes = np.atleast_1d(axes)
lstm_raw = PREDS[(fd, 'LSTM', s0)]['traj']; pg_copc = copc(PREDS[(fd, PROPOSED, s0)]['traj'], D['utf'], D['ctf'])
for ax, u in zip(axes, units):
    m = D['utf'] == u; c = D['ctf'][m]
    ax.plot(c, D['ytf'][m], 'k--', lw=2.0, label='Ground truth')
    ax.plot(c, lstm_raw[m], color='#d95f02', lw=1.2, alpha=0.85, label='Baseline LSTM')
    ax.plot(c, pg_copc[m], color='#1b9e77', lw=2.0, label='PG-DBN (COPC)')
    ax.set_title(f'{fd} engine #{u}'); ax.set_xlabel('Operating cycle'); ax.grid(True, ls=':', alpha=0.6)
axes[0].set_ylabel('Remaining useful life (cycles)'); axes[0].legend(loc='lower left')
plt.tight_layout(); plt.savefig(f'{FIG}/fig_trajectories_{fd}.pdf', bbox_inches='tight'); plt.savefig(f'{FIG}/fig_trajectories_{fd}.png', bbox_inches='tight'); plt.show()

In [ ]:
# Consistency bars for all nine models (paper Fig. 5 style), PG-DBN highlighted.
g = RESULTS[RESULTS.dataset == fd].groupby('model')
mvr = [g.get_group(m)['MVR_copc' if m == PROPOSED else 'MVR_raw'].mean() for m in MODEL_ORDER]
rgh = [g.get_group(m)['Rough_copc' if m == PROPOSED else 'Rough_raw'].mean() for m in MODEL_ORDER]
cols = ['#9e9e9e'] * len(BASELINES) + ['#1b9e77']
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, vals, lab in [(a1, mvr, 'Monotonicity violations, MVR (%)'), (a2, rgh, 'Trajectory roughness')]:
    ax.bar(MODEL_ORDER, vals, color=cols); ax.set_ylabel(lab); ax.grid(True, axis='y', ls=':', alpha=0.6)
    ax.set_xticks(range(len(MODEL_ORDER))); ax.set_xticklabels(MODEL_ORDER, rotation=35, ha='right')
a1.annotate(f'{mvr[-1]:.2f}%', (len(MODEL_ORDER) - 1, mvr[-1]), xytext=(0, 4), textcoords='offset points',
            ha='center', fontsize=9, fontweight='bold', color='#1b9e77')
plt.suptitle(f'{fd}: baselines are raw outputs, PG-DBN is the COPC output', fontsize=10)
plt.tight_layout(); plt.savefig(f'{FIG}/fig_consistency_{fd}.pdf', bbox_inches='tight'); plt.savefig(f'{FIG}/fig_consistency_{fd}.png', bbox_inches='tight'); plt.show()

In [ ]:
# NEW: ratchet-effect figure. Trajectory RMSE of each model before and after COPC, per subset.
fig, axes = plt.subplots(1, len(DATASETS), figsize=(4 * len(DATASETS), 3.6), sharey=False); axes = np.atleast_1d(axes)
x = np.arange(len(MODEL_ORDER))
for ax, fdx in zip(axes, DATASETS):
    r = RESULTS[RESULTS.dataset == fdx].groupby('model')[['TrajRMSE_raw', 'TrajRMSE_copc']].mean().loc[MODEL_ORDER]
    ax.bar(x - 0.2, r.TrajRMSE_raw, 0.4, color='#bdbdbd', label='raw output')
    ax.bar(x + 0.2, r.TrajRMSE_copc, 0.4, color=['#636363'] * len(BASELINES) + ['#1b9e77'], label='after COPC')
    ax.set_xticks(x); ax.set_xticklabels(MODEL_ORDER, rotation=60, ha='right', fontsize=8)
    ax.set_title(fdx); ax.grid(True, axis='y', ls=':', alpha=0.6)
axes[0].set_ylabel('Trajectory RMSE (cycles)'); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{FIG}/fig_ratchet_effect.pdf', bbox_inches='tight'); plt.savefig(f'{FIG}/fig_ratchet_effect.png', bbox_inches='tight'); plt.show()

In [ ]:
# Last-window error distribution, LSTM vs PG-DBN (seed 0), from stored predictions.
y = D['yte']; e_l = PREDS[(fd, 'LSTM', s0)]['last'] - y; e_p = PREDS[(fd, PROPOSED, s0)]['last'] - y
fig, ax = plt.subplots(figsize=(6, 3.5)); bins = np.linspace(-40, 40, 27)
ax.hist(e_l, bins=bins, alpha=0.5, color='#e41a1c', label=f'LSTM (score {nasa_score(e_l + y, y):.0f})')
ax.hist(e_p, bins=bins, alpha=0.6, color='#1b9e77', label=f'PG-DBN (score {nasa_score(e_p + y, y):.0f})')
ax.axvline(0, color='k', ls='--', lw=1); ax.set_xlabel('Prediction error (predicted - true RUL)'); ax.set_ylabel('Engines')
ax.legend(); ax.grid(True, ls=':', alpha=0.4)
plt.tight_layout(); plt.savefig(f'{FIG}/fig_error_distribution_{fd}.pdf', bbox_inches='tight'); plt.savefig(f'{FIG}/fig_error_distribution_{fd}.png', bbox_inches='tight'); plt.show()

## 12. Model size and CPU latency (Table IV)
Parameters, FP32 size, and median CPU latency for **one window** (batch 1, single thread, 1,000 runs after warm-up).
**Report the CPU name printed below** in the paper: Kaggle's CPU differs from the Apple M4 used for the current Table IV, so these numbers will differ.

In [ ]:
import platform
try:
    cpu_name = next((l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')), platform.processor())
except Exception:
    cpu_name = platform.processor()
torch.set_num_threads(1)
F_ = len(SENSORS); x1 = torch.randn(1, WINDOW, F_)
arch = {**{k: v(WINDOW, F_) for k, v in NN_MODELS.items()}, PROPOSED: PGDBN(WINDOW, F_)}
rows = []
for name, net in arch.items():
    net.eval(); n_par = sum(p.numel() for p in net.parameters() if p.requires_grad)
    with torch.no_grad():
        for _ in range(50): net(x1)
        ts = []
        for _ in range(1000):
            t0 = time.perf_counter(); net(x1); ts.append(time.perf_counter() - t0)
    rows.append(dict(Model=name, Params=n_par, Size_KB=round(n_par * 4 / 1024), Latency_ms=round(1000 * np.median(ts), 3),
                     Monotonic='Yes' if name == PROPOSED else 'No'))
torch.set_num_threads(os.cpu_count() or 1)
COMPLEX = pd.DataFrame(rows)
save_table(COMPLEX, 'table4_complexity', f'Model size and single-window CPU latency ({cpu_name}, 1 thread, FP32).')
print('CPU:', cpu_name)

## 13. Summary for the paper and download package

In [ ]:
lines = [f'Run finished in {(time.time() - T_START) / 60:.1f} min on {DEVICE}. Seeds: {SEEDS}. Subsets: {DATASETS}.', '']
for fdx in DATASETS:
    r = RESULTS[RESULTS.dataset == fdx]; p = r[r.model == PROPOSED]
    best_b = r[r.model != PROPOSED].groupby('model').RMSE.mean().idxmin()
    lines.append(f'{fdx}: PG-DBN RMSE {p.RMSE.mean():.2f} ± {p.RMSE.std(ddof=1) if len(p) > 1 else 0:.2f} '
                 f'(deployed after COPC {p.RMSE_dep.mean():.2f}); best baseline {best_b} '
                 f'{r[r.model == best_b].RMSE.mean():.2f}; PG-DBN trajectory RMSE raw {p.TrajRMSE_raw.mean():.2f} '
                 f'-> COPC {p.TrajRMSE_copc.mean():.2f}; raw MVR {p.MVR_raw.mean():.2f}% -> 0.00%')
lines += ['', 'Wilcoxon (last window), Holm-significant baselines where PG-DBN is better:']
for fdx in DATASETS:
    s = STATS[(STATS.Subset == fdx) & (STATS['Significant (Holm < 0.05)'] == 'yes') & (STATS.Diff > 0)].Baseline.tolist()
    lines.append(f'  {fdx}: {", ".join(s) if s else "none"}')
lines += ['', 'Ratchet test (trajectory after COPC), Holm-significant baselines where PG-DBN is better:']
for fdx in DATASETS:
    s = TRAJ_STATS[(TRAJ_STATS.Subset == fdx) & (TRAJ_STATS['Significant (Holm < 0.05)'] == 'yes')
                   & (TRAJ_STATS.TrajRMSE_COPC_baseline > TRAJ_STATS.TrajRMSE_COPC_PGDBN)].Baseline.tolist()
    lines.append(f'  {fdx}: {", ".join(s) if s else "none"}')
summary = '\n'.join(lines); print(summary)
open(f'{ROOT}/SUMMARY.txt', 'w').write(summary)
json.dump(dict(CFG=CFG, DATASETS=DATASETS, SEEDS=SEEDS, EPOCHS_BASE=EPOCHS_BASE, EPOCHS_PG=EPOCHS_PG,
               LAMBDA_MONO=LAMBDA_MONO, ABLATION_SUBSETS=ABLATION_SUBSETS, ABLATION_SEEDS=ABLATION_SEEDS,
               N_BOOT=N_BOOT, device=DEVICE, torch=torch.__version__, xgboost=xgb.__version__, lightgbm=lgb.__version__),
          open(f'{ROOT}/run_config.json', 'w'), indent=2)

archive = shutil.make_archive(ROOT, 'zip', ROOT)
print('\nDownload everything from:', archive)